# 10 — Synthetic Control Methods
**References:** Abadie & Gardeazabal (2003, *AER*) · Abadie, Diamond & Hainmueller (2010, *JASA*) · Abadie (2021, *Journal of Economic Literature*) · Athey & Imbens (2017): "the most important innovation in the policy evaluation literature in the last 15 years"

## Narrative thread
```
One treated unit -> Weighted donor pool -> Pre-treatment fit -> Post-treatment gap -> Placebo (permutation) inference
```

## The problem: comparative case studies

One aggregate unit gets treated — a state passes a law, a country suffers a shock
(California's tobacco tax in Prop 99; German reunification; terrorism in the Basque
Country). No single control unit is credible, and with $n = 1$ treated unit, DiD's
parallel-trends bet on any one comparison is heroic.

## The synthetic control

Build the counterfactual as a **convex combination of donor units** chosen to reproduce
the treated unit's *pre-treatment* trajectory:

$$\hat Y_{1t}(0) = \sum_{j=2}^{J+1} w_j^* Y_{jt}, \qquad
w_j \geq 0, \; \sum_j w_j = 1$$

with $\mathbf{w}^*$ minimizing pre-treatment discrepancy
$\lVert \mathbf{X}_1 - \mathbf{X}_0 \mathbf{w} \rVert$. The effect at time $t > T_0$ is the gap
$\hat\tau_{1t} = Y_{1t} - \hat Y_{1t}(0)$.

Why the constraints matter (Abadie 2021):
- **No extrapolation:** convex weights keep the counterfactual inside the donor data.
- **Transparency:** the weights *are* the comparison — you can name the donors.
- **Discipline:** weights are chosen on pre-treatment data only, never peeking at outcomes.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [ ]:
# ── Simulated Prop-99-style panel: 1 treated unit, 19 donors ─────────────
from scipy.optimize import minimize

T, T0 = 30, 20                       # 30 periods, treatment starts at t = 20
J = 19                               # donor units
rng = np.random.default_rng(42)

# Factor structure: units load differently on two common trends
factors = np.vstack([np.linspace(0, 4, T), np.sin(np.arange(T)/3)])
load_donors  = rng.uniform(0, 2, (J, 2))
load_treated = np.array([1.4, 1.1])              # inside the donors' convex hull

Y0_donors  = load_donors @ factors + rng.normal(0, 0.4, (J, T)) + rng.uniform(2, 8, (J, 1))
y_treated0 = load_treated @ factors + rng.normal(0, 0.4, T) + 5.0
TRUE_EFFECT = -2.5                                # policy REDUCES the outcome
y_treated = y_treated0.copy()
y_treated[T0:] += TRUE_EFFECT

# ── Fit weights on pre-treatment outcomes: min ||y1_pre - Y0_pre' w|| ────
def fit_synth(y1_pre, Y0_pre):
    Jn = Y0_pre.shape[0]
    def loss(w): return np.sum((y1_pre - w @ Y0_pre)**2)
    cons = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
    res = minimize(loss, np.full(Jn, 1/Jn), bounds=[(0, 1)]*Jn,
                   constraints=cons, method='SLSQP', options={'maxiter': 500})
    return res.x

w_star = fit_synth(y_treated[:T0], Y0_donors[:, :T0])
synth = w_star @ Y0_donors

top = np.argsort(w_star)[::-1][:5]
print('Donor weights (top 5):', ', '.join(f'unit {j}: {w_star[j]:.2f}' for j in top))
print(f'Pre-treatment RMSE: {np.sqrt(np.mean((y_treated[:T0] - synth[:T0])**2)):.3f}')
gap = y_treated - synth
print(f'Mean post-treatment gap: {gap[T0:].mean():.2f}   (true effect: {TRUE_EFFECT})')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(y_treated, color='#d62728', lw=2, label='treated unit')
axes[0].plot(synth, color='#1f77b4', lw=2, ls='--', label='synthetic control')
axes[0].axvline(T0 - 0.5, color='k', ls=':', lw=1)
axes[0].set_title('Treated vs synthetic'); axes[0].set_xlabel('period'); axes[0].legend()
axes[1].plot(gap, color='#444', lw=2)
axes[1].axvline(T0 - 0.5, color='k', ls=':', lw=1); axes[1].axhline(0, color='k', lw=1)
axes[1].axhline(TRUE_EFFECT, color='#d62728', ls='--', lw=1, label='true effect')
axes[1].set_title('Gap = treated − synthetic'); axes[1].set_xlabel('period'); axes[1].legend()
plt.tight_layout(); plt.show()

## Inference: in-place placebos

With one treated unit, classical standard errors make no sense. Abadie et al. (2010)
propose **permutation-style placebo tests**: apply the same synthetic-control machinery to
*every donor* as if it were treated. If the real treated unit's post/pre gap ratio is
extreme relative to the placebo distribution, the effect is unlikely to be noise.

The standard statistic is the ratio of post-treatment to pre-treatment RMSPE (root mean
squared prediction error); the permutation p-value is the rank of the treated unit's ratio.


In [ ]:
# ── Placebo-in-space inference ────────────────────────────────────────────
def rmspe(x): return np.sqrt(np.mean(x**2))

ratios, placebo_gaps = [], []
all_units = np.vstack([y_treated, Y0_donors])
for j in range(1, J + 1):                          # each donor as fake treated
    others = np.delete(all_units, j, axis=0)[1:]   # donors minus unit j (excl. real treated)
    wj = fit_synth(all_units[j, :T0], others[:, :T0])
    g  = all_units[j] - wj @ others
    placebo_gaps.append(g)
    ratios.append(rmspe(g[T0:]) / rmspe(g[:T0]))

ratio_treated = rmspe(gap[T0:]) / rmspe(gap[:T0])
pval = (1 + sum(r >= ratio_treated for r in ratios)) / (J + 1)

fig, ax = plt.subplots(figsize=(9, 4))
for g in placebo_gaps:
    ax.plot(g, color='#bbbbbb', lw=1)
ax.plot(gap, color='#d62728', lw=2.5, label='treated unit')
ax.axvline(T0 - 0.5, color='k', ls=':', lw=1); ax.axhline(0, color='k', lw=1)
ax.set_title(f'Placebo gaps: treated unit is extreme (permutation p = {pval:.3f})')
ax.set_xlabel('period'); ax.set_ylabel('gap'); ax.legend()
plt.tight_layout(); plt.show()

print(f'Post/pre RMSPE ratio, treated: {ratio_treated:.1f}')
print(f'Max ratio among {J} placebos:  {max(ratios):.1f}')
print(f'Permutation p-value:           {pval:.3f}')

## Practical guardrails (Abadie 2021)

1. **Good pre-treatment fit is non-negotiable** — a synthetic control that cannot track
   the pre-period cannot be trusted in the post-period.
2. **Long pre-treatment window** — protects against overfitting to noise.
3. **Restrict the donor pool** — donors must be unaffected by the treatment (no spillovers)
   and not hit by their own idiosyncratic shocks.
4. **No anticipation** — backdate $T_0$ if the policy was expected.
5. Report donor weights and placebo plots; discard placebos with terrible pre-fit before
   computing p-values.

## Key takeaways

| Concept | Statement |
|---|---|
| Setting | One (aggregate) treated unit, several potential comparison units |
| Synthetic control | Convex donor combination matching the pre-treatment path |
| Convexity | No extrapolation, transparent named weights |
| Effect | Post-treatment gap between treated and synthetic trajectories |
| Inference | Placebo-in-space permutation of the post/pre RMSPE ratio |
| Credibility | Lives or dies on pre-treatment fit and donor-pool hygiene |
